02_eda

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

OUT_DIR = Path("outputs")
stage1 = pd.read_pickle(OUT_DIR / "stage1_all_women.pkl")
stage2 = pd.read_pickle(OUT_DIR / "stage2_working_women.pkl")
outside = pd.read_pickle(OUT_DIR / "outside_lf_women.pkl")
fl = pd.read_pickle(OUT_DIR / "feature_lists.pkl")
CAT, NUM = list(fl["FEATURES_CAT"]), list(fl["FEATURES_NUM"])
VL = pd.read_pickle(OUT_DIR / "value_labels.pkl")

SRC = {"sector": "b1q3_perv1", "social_group": "b3q4_hhv1", "religion": "b3q3_hhv1",
       "state": "state_perv1", "marital": "b4q7_perv1"}

def lab(series, var):
    """Turn codes into SPSS value labels where they exist; otherwise keep the code."""
    m = {}
    for k, v in VL.get(SRC.get(var, var), {}).items():
        try:
            m[str(int(float(k)))] = v
        except Exception:
            m[str(k)] = v
    s = series.astype(str).str.replace(r"\.0$", "", regex=True)
    return s.map(m).fillna(s)

print(stage1.shape, stage2.shape, outside.shape)

In [ ]:
stage1["kids_u6_band"] = stage1["hh_kids_u6"].clip(upper=3).astype(int).astype(str).replace({"3": "3+"})
stage1["age_band"] = pd.cut(stage1["age"], [14, 24, 34, 44, 54, 59],
                            labels=["15-24", "25-34", "35-44", "45-54", "55-59"])

def part_table(col):
    key = lab(stage1[col], col) if col in SRC else stage1[col]
    return (stage1.groupby(key, observed=True)["lfp"]
            .agg(participation="mean", n="size").round(3))

tables = {}
for col in ["edu_group", "sector", "social_group", "marital", "religion", "kids_u6_band", "age_band"]:
    tables[col] = part_table(col)
    print(f"\n--- Participation by {col}")
    display(tables[col])

st = part_table("state").sort_values("participation")
print("\n--- Lowest 5 and highest 5 states")
display(pd.concat([st.head(5), st.tail(5)]))

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.ravel(), ["edu_group", "sector", "social_group", "marital", "kids_u6_band", "age_band"]):
    tables[col]["participation"].plot.bar(ax=ax, color="#1F3A5F")
    ax.set_title(f"Participation by {col}"); ax.set_ylabel("Share in labour force"); ax.set_xlabel("")
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_eda_participation.png", dpi=150); plt.show()

In [ ]:
prof = stage2.groupby("own_economy_type").agg(
    n=("HHID", "size"), mean_age=("age", "mean"), mean_edu_years=("edu_years", "mean"),
    mean_log_pc_exp=("log_pc_exp", "mean"), mean_kids_u6=("hh_kids_u6", "mean"))
prof["rural_share"] = stage2.assign(r=(stage2["sector"] == "1")).groupby("own_economy_type")["r"].mean()
display(prof.round(3))

for col in ["edu_group", "social_group", "marital"]:
    key = lab(stage2[col], col) if col in SRC else stage2[col]
    print(f"\n--- {col}: share of each economy type within each group (row %)")
    display(pd.crosstab(key, stage2["own_economy_type"], normalize="index").round(3))

prof.to_csv(OUT_DIR / "eda_economy_profile.csv")

In [ ]:
work = stage1[stage1["lfp_group"] == "working"]
rows = []
for c in NUM:
    lo, hi = work[c].quantile([0.01, 0.99])
    sd = np.sqrt((work[c].var() + outside[c].var()) / 2)
    rows.append({"feature": c,
                 "mean_working": work[c].mean(), "mean_outside": outside[c].mean(),
                 "std_diff": (outside[c].mean() - work[c].mean()) / sd if sd > 0 else np.nan,
                 "share_outside_beyond_working_1_99pct": ((outside[c] < lo) | (outside[c] > hi)).mean()})
support = pd.DataFrame(rows).round(3)
display(support)
support.to_csv(OUT_DIR / "eda_common_support.csv", index=False)

for c in CAT:
    unseen = set(outside[c].dropna().unique()) - set(work[c].dropna().unique())
    print(c, "- categories among non-participants never seen in working women:", unseen or "none")